# Colab GPU Chat UI (HF形式そのまま)

- GGUF不要で、Colab GPU上でチャット推論を行うための最小構成です。
- `MODEL_SOURCE` を切り替えることで、以下の3パターンに対応します。
  - `base`: ベースモデル
  - `merged`: すでにマージ済みモデル
  - `adapter_merge`: ベース + LoRAアダプタをその場でマージ

In [ ]:
# 必要なときだけ True にして実行してください（実行後はランタイム再起動）
# vLLMチャット用途の最小セット（StructEval依存なし）
RUN_INSTALL = True

if RUN_INSTALL:
    import subprocess

    cmds = [
        # 競合しやすい主要パッケージを先に外してから最小セットを入れる
        'pip uninstall -y protobuf huggingface_hub transformers tokenizers vllm',
        'pip install --no-cache-dir "protobuf==5.29.3"',
        'pip install --no-cache-dir '
        '"torch==2.9.0" '
        '"triton==3.5.0" '
        '"huggingface-hub>=0.36.0" '
        '"tokenizers>=0.22.0" '
        '"vllm>=0.13.0" '
        '"gradio>=4.0.0" '
        '"accelerate" '
        '"peft"',
        # Qwen3.5(qwen3_5)はtransformersメインラインが必要な場合がある
        'pip install --no-cache-dir --upgrade "git+https://github.com/huggingface/transformers.git"',
        'python3 -c "import google.protobuf, vllm, transformers, huggingface_hub, gradio; '
        "from transformers.models.auto.configuration_auto import CONFIG_MAPPING; "
        "print('protobuf', google.protobuf.__version__); "
        "print('vllm', vllm.__version__); "
        "print('transformers', transformers.__version__); "
        "print('huggingface_hub', huggingface_hub.__version__); "
        "print('qwen3_5_supported', 'qwen3_5' in CONFIG_MAPPING)" '"',
    ]
    for c in cmds:
        subprocess.check_call(c, shell=True)
    print("✅ setup finished. Please restart runtime now.")



In [ ]:
import os

# -----------------------------
# Config
# -----------------------------
MODEL_SOURCE = "base"  # "base" | "merged" | "adapter_merge"

# HF形式モデルID
BASE_MODEL_ID = "Qwen/Qwen3.5-4B"
MERGED_MODEL_ID = ""
ADAPTER_ID = ""

# 推論設定（Transformers fallback向けに控えめ推奨）
MAX_NEW_TOKENS = 512
MAX_INPUT_TOKENS = 2048
MAX_HISTORY_TURNS = 3
TEMPERATURE = 0.0
TOP_P = 1.0

# tool-call 実験設定
TASK_UI_DELAY_SEC = 0.35
TOOL_CALL_MAX_STEPS = 4
TOOL_CALL_RETRY = 1

# merge一時保存先（adapter_merge時）
MERGED_LOCAL_DIR = "/content/merged_for_chat"



In [ ]:
import gc
import json
from pathlib import Path

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
import google.protobuf
import torch


def detect_model_type(path_or_repo: str):
    # Local merged dir を優先して config.json を読む
    local_cfg = Path(path_or_repo) / "config.json"
    if local_cfg.exists():
        return json.loads(local_cfg.read_text()).get("model_type")

    # HF repo の場合はキャッシュ済み config.json を読む
    from huggingface_hub import hf_hub_download

    cfg_path = hf_hub_download(path_or_repo, filename="config.json")
    return json.loads(Path(cfg_path).read_text()).get("model_type")


def validate_transformers_arch_support(path_or_repo: str):
    model_type = detect_model_type(path_or_repo)
    if model_type is None:
        print("[WARN] model_type could not be detected from config.json")
        return None

    if model_type not in CONFIG_MAPPING:
        import transformers

        raise RuntimeError(
            f"Installed transformers={transformers.__version__} does not support model_type={model_type}. "
            "Run install cell (RUN_INSTALL=True) and restart runtime."
        )

    print(f"[INFO] model_type={model_type} is supported by transformers")
    return model_type


def resolve_model_path_and_tokenizer():
    if MODEL_SOURCE == "base":
        model_id = BASE_MODEL_ID
        print(f"[INFO] Using base model: {model_id}")
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        return model_id, tokenizer

    if MODEL_SOURCE == "merged":
        model_id = MERGED_MODEL_ID
        print(f"[INFO] Using merged model: {model_id}")
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        return model_id, tokenizer

    if MODEL_SOURCE == "adapter_merge":
        from peft import PeftModel

        print(f"[INFO] Loading base model for merge: {BASE_MODEL_ID}")
        # vLLM初期化時のGPU競合を避けるため、マージはCPU上で実行
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="cpu",
            trust_remote_code=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

        print(f"[INFO] Loading adapter: {ADAPTER_ID}")
        lora_model = PeftModel.from_pretrained(base_model, ADAPTER_ID)

        print("[INFO] Merging adapter...")
        merged_model = lora_model.merge_and_unload()

        os.makedirs(MERGED_LOCAL_DIR, exist_ok=True)
        merged_model.save_pretrained(MERGED_LOCAL_DIR)
        tokenizer.save_pretrained(MERGED_LOCAL_DIR)

        del base_model, lora_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"[INFO] Merged model saved to: {MERGED_LOCAL_DIR}")
        return MERGED_LOCAL_DIR, tokenizer

    raise ValueError("MODEL_SOURCE must be one of: base, merged, adapter_merge")


model_path, tokenizer = resolve_model_path_and_tokenizer()
print(f"[INFO] Model path resolved: {model_path}")
model_type = validate_transformers_arch_support(model_path)

# 本番環境合わせ: protobuf 5.29.3 を期待
pb_ver = google.protobuf.__version__
if pb_ver != "5.29.3":
    raise RuntimeError(
        f"Incompatible protobuf version: {pb_ver}. "
        "Please run install cell and restart runtime."
    )

if torch.cuda.is_available():
    # L4でのTransformers推論速度改善
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

# vLLMはimport前に環境変数を設定
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_LOGGING_LEVEL"] = "INFO"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "1"

# tokenizerはベース側を明示（mergedフォルダのtokenizer由来不整合を回避）
tokenizer_path_for_vllm = BASE_MODEL_ID if MODEL_SOURCE == "adapter_merge" else model_path

# vLLM起動フォールバック（OOM/初期化失敗を段階的に回避）
_try_cfgs = [
    {"max_model_len": 4096, "gpu_memory_utilization": 0.85},
    {"max_model_len": 3072, "gpu_memory_utilization": 0.80},
    {"max_model_len": 2048, "gpu_memory_utilization": 0.72},
]
llm = None
vllm_available = False
fallback_transformers_model = None
last_err = None
SamplingParams = None

# qwen3_5 + latest transformers では vllm との組み合わせが壊れることがあるため安全にフォールバック
vllm_import_ok = False
try:
    from vllm import LLM, SamplingParams
    vllm_import_ok = True
except Exception as e:
    last_err = e
    print(f"[WARN] vLLM import failed: {type(e).__name__}: {e}")

if vllm_import_ok:
    for i, cfg in enumerate(_try_cfgs, 1):
        try:
            print(
                f"[INFO] vLLM init try {i}/{len(_try_cfgs)} "
                f"(max_model_len={cfg['max_model_len']}, gpu_mem={cfg['gpu_memory_utilization']})"
            )
            llm = LLM(
                model=model_path,
                tokenizer=tokenizer_path_for_vllm,
                trust_remote_code=True,
                tensor_parallel_size=1,
                enforce_eager=True,
                max_model_len=cfg["max_model_len"],
                gpu_memory_utilization=cfg["gpu_memory_utilization"],
                disable_log_stats=True,
            )
            print("[INFO] vLLM loaded.")
            vllm_available = True
            break
        except Exception as e:
            last_err = e
            print(f"[WARN] vLLM init failed on try {i}: {type(e).__name__}: {e}")

if not vllm_available:
    print(f"[WARN] Using Transformers backend. vLLM last_error={last_err}")
    fallback_transformers_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True,
    )
    fallback_transformers_model.eval()
    print(
        "[INFO] Transformers fallback model loaded on device:",
        getattr(fallback_transformers_model, "device", "unknown"),
    )

if torch.cuda.is_available():
    print("[INFO] CUDA is available:", torch.cuda.get_device_name(0))
else:
    print("[WARN] CUDA is NOT available. Running on CPU.")



In [ ]:
import json
import re
import time
import gradio as gr

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search a target object in the environment and return execute.",
            "parameters": {
                "type": "object",
                "properties": {
                    "instruct": {"type": "string", "description": "Instruction for search"},
                },
                "required": ["instruct"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "go_to",
            "description": "Move step trigger. No parameter is required because destination is computed externally.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "action",
            "description": "Execute a VLA action and return execute.",
            "parameters": {
                "type": "object",
                "properties": {
                    "instruct": {"type": "string", "description": "Action instruction"},
                },
                "required": ["instruct"],
            },
        },
    },
]


def _strip_thinking(text: str) -> str:
    return re.sub(r"<think>[\s\S]*?</think>", "", text).strip()


def _strip_tool_markup(text: str) -> str:
    text = re.sub(r"<tool_call>[\s\S]*?</tool_call>", "", text)
    text = re.sub(r"<function=.*?</function>", "", text, flags=re.DOTALL)
    return text.strip()


def _parse_tool_calls(text: str):
    calls = []
    for m in re.finditer(r"<function=([a-zA-Z0-9_\-]+)>([\s\S]*?)</function>", text):
        name = m.group(1).strip()
        body = m.group(2)
        args = {}
        for p in re.finditer(r"<parameter=([a-zA-Z0-9_\-]+)>([\s\S]*?)</parameter>", body):
            args[p.group(1).strip()] = p.group(2).strip()
        calls.append({"name": name, "args": args})
    return calls


def _required_cycle_count(user_text: str) -> int:
    t = user_text.lower().strip()
    if not t:
        return 1

    clauses = [c.strip() for c in re.split(r"\b(?:and then|then|after that|after|before|and)\b|[,.;]", t) if c.strip()]
    return max(1, len(clauses))


def _is_tool_plan_sufficient(tool_calls, user_text: str):
    cycles = _required_cycle_count(user_text)
    req = cycles * 3
    if len(tool_calls) < req:
        return False, f"too few tool calls: got={len(tool_calls)} required>={req}"

    expected = ["search", "go_to", "action"]
    for i in range(req):
        name = tool_calls[i].get("name", "")
        if name != expected[i % 3]:
            return False, f"invalid order at {i}: expected={expected[i % 3]} got={name}"
        ins = tool_calls[i].get("args", {}).get("instruct", "").strip()
        if name != "go_to" and not ins:
            return False, f"empty instruct at {i}"

    return True, "ok"


def _force_cycle_plan(user_text: str):
    clauses = [c.strip() for c in re.split(r"\b(?:and then|then|after that|after|before|and)\b|[,.;]", user_text.strip()) if c.strip()]
    if not clauses:
        clauses = [user_text.strip()]

    calls = []
    for clause in clauses:
        calls.append({"name": "search", "args": {"instruct": f"search target for: {clause}"}})
        calls.append({"name": "go_to", "args": {}})
        calls.append({"name": "action", "args": {"instruct": clause}})
    return calls


def _render_task_queue(tasks):
    if not tasks:
        return "### Task Queue\n- (empty)"
    lines = ["### Task Queue"]
    for i, t in enumerate(tasks, 1):
        ins = t.get("args", {}).get("instruct", "")
        lines.append(f"- {i}. `{t['name']}` instruct=`{ins}`")
    return "\n".join(lines)


def _render_tool_call_log(tool_calls, raw_text=""):
    lines = ["### Tool Calls"]
    if not tool_calls:
        lines.append("- (none)")
    else:
        for i, c in enumerate(tool_calls, 1):
            ins = c.get("args", {}).get("instruct", "")
            lines.append(f"- {i}. `{c.get('name', '')}` instruct=`{ins}`")
    if raw_text:
        lines.append("\nRaw model output:")
        lines.append("```text")
        lines.append(raw_text)
        lines.append("```")
    return "\n".join(lines)


def _build_prompt(messages):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tools=TOOLS,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        try:
            return tokenizer.apply_chat_template(
                messages,
                tools=TOOLS,
                tokenize=False,
                add_generation_prompt=True,
            )
        except TypeError:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )


def _generate_text(prompt: str) -> str:
    if vllm_available:
        sampling = SamplingParams(
            max_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
        )
        outs = llm.generate([prompt], sampling)
        txt = outs[0].outputs[0].text if outs and outs[0].outputs else ""
        return _strip_thinking(txt.strip())

    import torch

    tok = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS)
    inputs = {k: v.to(fallback_transformers_model.device) for k, v in tok.items()}
    with torch.no_grad():
        outputs = fallback_transformers_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=(TEMPERATURE > 0.0),
            temperature=TEMPERATURE,
            top_p=TOP_P,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    txt = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return _strip_thinking(txt)


def _execute_tool(call):
    return {
        "tool": call["name"],
        "args": call.get("args", {}),
        "status": "executed",
        "result": "execute",
    }


def _build_tool_forced_user_message(user_text: str, reason: str = ""):
    reason_line = f"Previous output failed: {reason}\n" if reason else ""
    return (
        "Decompose and call tools for this instruction:\n"
        f"INSTRUCTION: {user_text.strip()}\n"
        "Output ONLY tool_call tags. No prose.\n"
        "Each step must be atomic and ordered.\n"
        "Use only function names: search, go_to, action.\n"
        "For EVERY clause, output exactly this strict triplet in order: search -> go_to -> action.\n"
        "Repeat the triplet for all clauses in sequence.\n"
        "Use parameter `instruct` for search/action only.\nFor go_to, output only <function=go_to></function> (no parameter).\n"
        "Format: <tool_call><function=NAME><parameter=instruct>...</parameter></function></tool_call>\n"
        f"{reason_line}"
    )


FEWSHOT_USER = "pick an apple and place it on the shelf"
FEWSHOT_ASSISTANT = """<tool_call>
<function=search>
<parameter=instruct>
search an apple
</parameter>
</function>
</tool_call>
<tool_call>
<function=go_to>
</function>
</tool_call>
<tool_call>
<function=action>
<parameter=instruct>
pick up an apple
</parameter>
</function>
</tool_call>
<tool_call>
<function=search>
<parameter=instruct>
search shelf
</parameter>
</function>
</tool_call>
<tool_call>
<function=go_to>
</function>
</tool_call>
<tool_call>
<function=action>
<parameter=instruct>
place an apple on the shelf
</parameter>
</function>
</tool_call>"""


def _tool_response_content(exec_result):
    return (
        "<tool_response>"
        f"<function={exec_result['tool']}>"
        f"{json.dumps(exec_result, ensure_ascii=False)}"
        "</function>"
        "</tool_response>"
    )


def run_agent(user_text, chat_history, state_messages):
    if not user_text or not user_text.strip():
        yield chat_history, state_messages, _render_task_queue([]), _render_tool_call_log([]), ""
        return

    chat_history = list(chat_history) if chat_history else []

    history_pairs = []
    for i in range(0, len(chat_history), 2):
        if i + 1 < len(chat_history):
            history_pairs.append((chat_history[i].get("content", ""), chat_history[i + 1].get("content", "")))
    if MAX_HISTORY_TURNS > 0:
        history_pairs = history_pairs[-MAX_HISTORY_TURNS:]

    msgs = [{
        "role": "system",
        "content": (
            "You are a robot execution agent. Always output tool calls first. "
            "Use only tools: search, go_to, action. "
            "Use parameter=instruct for search/action; go_to requires no parameter. "
            "Output format must be <tool_call><function=...><parameter=instruct>...</parameter></function></tool_call>. "
            "Decompose into atomic subtasks. Do not collapse multi-step tasks into one action. "
            "For every clause, output exactly 3 steps in this strict order: search -> go_to -> action. "
            "Repeat this triplet for all clauses in order. "
            "Use concise, non-empty instruct strings per step. "
            "Do not provide final answer before tool calls. Do not output hidden reasoning."
        ),
    }]

    # one-shot example to stabilize strict format
    msgs.append({"role": "user", "content": _build_tool_forced_user_message(FEWSHOT_USER)})
    msgs.append({"role": "assistant", "content": FEWSHOT_ASSISTANT})

    for u, a in history_pairs:
        msgs.append({"role": "user", "content": u})
        msgs.append({"role": "assistant", "content": a})
    msgs.append({"role": "user", "content": _build_tool_forced_user_message(user_text)})

    tool_calls = []
    last_model_text = ""

    for attempt in range(TOOL_CALL_RETRY + 1):
        prompt = _build_prompt(msgs)
        model_text = _generate_text(prompt)
        last_model_text = model_text
        tool_calls = _parse_tool_calls(model_text)

        ok, reason = _is_tool_plan_sufficient(tool_calls, user_text) if tool_calls else (False, "no tool_call")
        if tool_calls and ok:
            msgs.append({"role": "assistant", "content": model_text})
            break

        if attempt < TOOL_CALL_RETRY:
            msgs.append({"role": "assistant", "content": model_text})
            msgs.append({
                "role": "user",
                "content": _build_tool_forced_user_message(user_text, reason),
            })

    ok_final, reason_final = _is_tool_plan_sufficient(tool_calls, user_text) if tool_calls else (False, "no tool_call")
    if (not tool_calls) or (not ok_final):
        tool_calls = _force_cycle_plan(user_text)
        last_model_text = f"[FORCED_CYCLE] reason={reason_final}\n" + last_model_text

    pending = list(tool_calls)
    tool_log = _render_tool_call_log(tool_calls, last_model_text)
    yield chat_history, state_messages, _render_task_queue(pending), tool_log, ""

    executed = []
    while pending:
        current = pending.pop(0)
        result = _execute_tool(current)
        executed.append(result)
        msgs.append({"role": "tool", "content": _tool_response_content(result)})
        yield chat_history, state_messages, _render_task_queue(pending), tool_log, ""
        time.sleep(TASK_UI_DELAY_SEC)

    final_prompt = _build_prompt(msgs)
    final_text = _strip_tool_markup(_generate_text(final_prompt))
    if not final_text:
        final_text = f"Issued {len(tool_calls)} tool_call(s) and executed all."

    chat_history.append({"role": "user", "content": user_text})
    chat_history.append({"role": "assistant", "content": final_text})

    state_messages = list(state_messages) if state_messages else []
    state_messages.append({
        "user": user_text,
        "tool_calls": tool_calls,
        "executed": executed,
        "raw_model_text": last_model_text,
    })

    yield chat_history, state_messages, _render_task_queue([]), tool_log, ""


with gr.Blocks() as demo:
    gr.Markdown("## Colab GPU Chat (Direct tool_call demo)")
    chatbot = gr.Chatbot(type="messages", label="Chat")
    task_queue = gr.Markdown(_render_task_queue([]))
    tool_call_log = gr.Markdown(_render_tool_call_log([]))
    user_input = gr.Textbox(label="Instruction", placeholder="pick an apple and place it on the shelf")
    state_messages = gr.State([])

    user_input.submit(
        run_agent,
        inputs=[user_input, chatbot, state_messages],
        outputs=[chatbot, state_messages, task_queue, tool_call_log, user_input],
    )

# share=True で外部URL発行
demo.launch(share=True, debug=False)









